# ⚡ Atelier 3 — Le streaming

⏱️ **Durée : 30 à 35 minutes** · Niveau : débutant · Modèle : Mistral via LangChain

<img src="./assets/LC_streaming.png" width="420">

Attendre une réponse complète peut sembler long. Le **streaming** affiche la réponse **au fil de l'eau**, ce qui réduit la latence *perçue* et rend l'agent bien plus agréable à utiliser.

### 🎯 À la fin, vous saurez

- expliquer la différence entre `invoke` (tout d'un coup) et `stream` (au fil de l'eau) ;
- utiliser `stream_mode="values"` pour suivre l'**état** étape par étape ;
- utiliser `stream_mode="messages"` pour afficher la réponse **token par token** ;
- émettre des données personnalisées **depuis un outil** avec `get_stream_writer`.

> **Prérequis :** les Ateliers 1 et 2 aident, mais chaque section est autonome.

## 🧠 `invoke` vs `stream` : la lettre finie ou l'écriture en direct

- **`invoke`** = on vous remet une **lettre déjà rédigée** : rien ne s'affiche tant qu'elle n'est pas finie.
- **`stream`** = vous regardez quelqu'un **écrire en direct** : les mots apparaissent au fur et à mesure.

Le temps de calcul total est similaire ; c'est la latence **perçue** qui change radicalement.

```text
invoke :  [......... attente .........] → réponse complète d'un coup
stream :  ré → répo → répon → réponse ...  (affichage progressif)
```

## 🛠️ 0. Préparer Mistral et un agent de démonstration

Chargement de `.env` (sans afficher de secret), puis [`ChatMistralAI`](https://docs.langchain.com/oss/python/integrations/chat/mistralai) (`mistral-medium-latest`, `temperature=0`) et un petit agent humoriste réutilisé dans les démos.

In [1]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Charge les variables d'environnement depuis le fichier .env
load_dotenv()

# Vérification de la présence de la clé API Mistral
if not os.getenv("MISTRAL_API_KEY"):
    raise ValueError("❌ MISTRAL_API_KEY non trouvée. Créez un fichier .env avec MISTRAL_API_KEY=votre_clé")
if not os.getenv("MISTRAL_SERVER_URL"):
    raise ValueError("❌ MISTRAL_SERVER_URL non trouvée. Créez un fichier .env avec MISTRAL_SERVER_URL=votre_url")

print("✅ Clé API Mistral chargée.")
print("✅ URL du serveur Mistral chargée.")

# Vérification des variables d'environnement et des packages requis
from util.env_utils import doublecheck_env
doublecheck_env("example.env")  # vérification des variables de l'environment

🔒 Sortie masquée à la publication (contenait une valeur d'environnement ou un chemin local).


In [ ]:
from langchain_mistralai import ChatMistralAI
from langgraph.prebuilt import create_react_agent

# Nom du modèle — une seule constante pour garder la cohérence entre les notebooks.
# Note : le serveur dédié n'expose PAS mistral-large-latest.
# mistral-medium-latest est le modèle de génération le plus capable disponible ici.
MODEL = "mistral-medium-latest"

# Création du modèle Mistral que LangChain utilisera pour générer les réponses.
# temperature=0 garantit des réponses déterministes (utile pour le SQL).
# base_url (alias de endpoint) pointe vers le serveur dédié — MISTRAL_SERVER_URL inclut /v1.
def _normaliser_endpoint(url: str) -> str:
    """Retourne une URL de base terminée par /v1, sans afficher sa valeur."""
    base = (url or "").strip().rstrip("/")
    return base if base.endswith("/v1") else f"{base}/v1"


llm = ChatMistralAI(
    model=MODEL,
    temperature=0,
    api_key=os.getenv("MISTRAL_API_KEY"),
    endpoint=_normaliser_endpoint(os.getenv("MISTRAL_SERVER_URL")),
)

print(f"✅ Modèle {MODEL} initialisé.")

agent = create_react_agent(
    model=llm,
    prompt="Tu es un humoriste spécialiste du développement logiciel. Tu réponds en français.",
)

print(f"✅ Agent avec {MODEL} initialisé.")

> 👀 **Résultat attendu :** `✅ Modèle mistral-medium-latest initialisé.` puis `✅ Agent avec mistral-medium-latest initialisé.`

## 1. Sans streaming — `invoke`

> 🔮 **Pause prédiction :** avant l'exécution, à quel moment le texte apparaîtra-t-il : progressivement, ou d'un seul bloc à la fin ?

In [3]:
# invoke() attend la réponse complète avant de retourner.
# Toute la latence est concentrée ici — on ne voit rien jusqu'à la fin.
result = agent.invoke({"messages": [{"role": "user", "content": "Raconte-moi une blague"}]})
print(result["messages"][1].content)

Bien sûr ! En voici une pour les développeurs :

**Pourquoi les développeurs confondent-ils toujours Halloween et Noël ?**
*Parce que OCT 31 == DEC 25 !* 🎃🎄

*(Explication pour les non-inités : en octal (base 8), 31 = 25 en décimal (base 10).)*

Et une autre pour la route :
**Combien de développeurs faut-il pour changer une ampoule ?**
*Aucun, c'est un problème matériel !* 💡😄

Tu en veux d'autres ?


> 🔍 **Lecture :** rien ne s'affiche pendant le calcul, puis la blague apparaît **d'un coup**. Toute la latence est concentrée à la fin.

## 2. Streaming — mode `values`

`stream_mode="values"` renvoie l'**état complet** à chaque étape de l'agent. Pratique pour suivre la progression d'une conversation (utile quand des outils entrent en jeu).

In [4]:
# stream_mode="values" retourne l'état complet à chaque étape de l'agent.
# C'est utile pour voir la progression de la conversation.
for step in agent.stream(
    {"messages": [{"role": "user", "content": "Raconte-moi une blague de développeur"}]},
    stream_mode="values",
):
    # À chaque étape, on affiche uniquement le dernier message
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Raconte-moi une blague de développeur


================================== Ai Message ==================================

Bien sûr ! En voici une qui devrait te faire sourire (ou grogner, selon ton niveau de caféine) :

**Pourquoi les développeurs confondent-ils toujours Halloween et Noël ?**
*Parce que OCT 31 == DEC 25 !*

*(Explication pour les non-initiés : en informatique, OCT c'est l'octal, DEC le décimal... et 31 en octal = 25 en décimal. Oui, on est des gens drôles.)*

Tu en veux une autre ? 😄


> 🔍 **Lecture :** on voit l'état évoluer étape par étape. Ici l'agent n'a pas d'outil : on observe surtout l'apparition de l'`AIMessage`.

## 3. Streaming — mode `messages` (token par token)

`stream_mode="messages"` renvoie les **tokens un par un**. C'est le mode qui donne l'effet « machine à écrire ».

> 🔮 **Pause prédiction :** qui produit ces tokens — LangChain, Python, ou **Mistral** ?

In [5]:
# stream_mode="messages" retourne les tokens un par un.
# end="" supprime le retour à la ligne entre chaque token.
# flush=True force l'affichage immédiat du token.
for token, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "Écris-moi un poème court sur les bugs informatiques."}]},
    stream_mode="messages",
):
    print(f"{token.content}", end="")
    # print(f"{token.content}", end="", flush=True)

**"Ode au Bug Éternel"**

*Ô toi, bug

 malicieux,*
*Qui dans mon code es si heureux,*
*Tu

 danses entre les lignes,*
*Et fais planter mes cygnes

.*

*Je te cherche, je te traque,*
*Mais tu

 ris, coquin, en cache.*
*Un point-virgule

 oublié,*
*Et c’est l’appli qui a pété !*



*Un jour, peut-être, je te tuerai,*
*Mais

 en attendant…*
*… je redémarre.* 😤

💻

> 🔍 **Lecture :** le texte s'affiche progressivement. Les tokens proviennent de **Mistral** ; `end=""` les colle bout à bout. C'est exactement ce que voit un utilisateur d'application de chat.

## 4. Streaming **depuis un outil** (`custom`)

Un outil peut émettre ses propres messages de progression pendant qu'il travaille, grâce à [`get_stream_writer`](https://docs.langchain.com/oss/python/langchain/streaming). On lit alors deux flux : l'état (`values`) et les données personnalisées (`custom`).

In [ ]:
from langgraph.prebuilt import create_react_agent
from langgraph.config import get_stream_writer


def get_meteo(ville: str) -> str:
    """Obtenir la météo pour une ville donnée."""
    # get_stream_writer() permet d'émettre des données pendant l'exécution de l'outil
    writer = get_stream_writer()
    writer(f"🔍 Recherche des données pour : {ville}")
    writer(f"✅ Données récupérées pour : {ville}")
    # Retourne le résultat (simulé ici)
    return f"Il fait toujours beau à {ville} !"


# Agent avec un outil qui utilise le streaming custom
agent_meteo = create_react_agent(
    model=llm,
    tools=[get_meteo],
    prompt="Tu es un assistant météo. Tu réponds en français.",
)

# stream_mode=["values", "custom"] reçoit les deux flux simultanément
for chunk in agent_meteo.stream(
    {"messages": [{"role": "user", "content": "Quelle est la météo à Paris ?"}]},
    stream_mode=["values", "custom"],
):
    print(chunk)

In [7]:
# On peut filtrer pour ne voir que les données custom (de l'outil)
for chunk in agent_meteo.stream(
    {"messages": [{"role": "user", "content": "Quelle est la météo à Lyon ?"}]},
    stream_mode=["custom"],
):
    print(chunk[1])

🔍 Recherche des données pour : Lyon
✅ Données récupérées pour : Lyon


🔍 Recherche des données pour : Lyon
✅ Données récupérées pour : Lyon


🔍 Recherche des données pour : Lyon
✅ Données récupérées pour : Lyon


🔍 Recherche des données pour : Lyon
✅ Données récupérées pour : Lyon


🔍 Recherche des données pour : Lyon
✅ Données récupérées pour : Lyon


🔍 Recherche des données pour : Lyon
✅ Données récupérées pour : Lyon


> 🔍 **Qui fait quoi ?**
>
> | Acteur | Rôle |
> |---|---|
> | L'outil (Python) | émet « 🔍 Recherche… / ✅ Données… » via `get_stream_writer`. |
> | LangGraph | achemine ces messages dans le flux `custom`. |
> | Mistral | décide d'appeler l'outil puis rédige la réponse finale. |

## 5. Résumé des modes de streaming

| Mode | Ce qu'il renvoie | Usage typique |
|---|---|---|
| `values` | l'**état complet** à chaque étape | suivre la progression, le débogage |
| `messages` | les **tokens** un par un | effet « machine à écrire » côté UI |
| `custom` | les **données émises par les outils** | barres de progression, journaux d'outil |

🧠 On peut combiner les modes, par exemple `stream_mode=["values", "custom"]`.

## 🧪 Micro-exercice — Comparer les modes

Reprenez l'agent météo et **changez le `stream_mode`** pour observer la différence entre les flux.

### ✅ Critères de réussite
- vous modifiez le paramètre `stream_mode` ;
- vous décrivez ce que chaque mode affiche ;
- vous isolez au moins une fois le flux `custom` de l'outil.

💡 Le squelette est **commenté** pour que « Run All » fonctionne avant votre essai.

In [8]:
# 👉 À vous : décommentez et changez le stream_mode pour comparer.
# for chunk in agent_meteo.stream(
#     {"messages": [{"role": "user", "content": "Quelle est la météo à Marseille ?"}]},
#     stream_mode=["values", "custom"],  # ← essayez ["custom"], ["messages"], ["values"]
# ):
#     if chunk[0] == "custom":
#         print(chunk[1])

<details>
<summary>✅ Voir une correction possible</summary>

```python
for chunk in agent_meteo.stream(
    {"messages": [{"role": "user", "content": "Quelle est la météo à Marseille ?"}]},
    stream_mode=["values", "custom"],
):
    if chunk[0] == "custom":
        print(chunk[1])
```

En filtrant `chunk[0] == "custom"`, on n'affiche que les messages de progression émis par l'outil `get_meteo`. Retirez le filtre pour voir aussi l'état (`values`).
</details>

## 🧭 Ce qu'il faut retenir

- ✅ `invoke` renvoie la réponse **complète** ; `stream` l'affiche **progressivement** ;
- ✅ `values` = l'état à chaque étape ; `messages` = les tokens un par un ;
- ✅ un outil peut émettre sa propre progression via `get_stream_writer` (flux `custom`) ;
- ✅ le streaming réduit la latence **perçue**, pas le temps de calcul total ;
- ✅ on peut combiner plusieurs modes.

### 🧭 Transition vers L4
Nos agents savent **agir**, **dialoguer** et **répondre en direct**. Dans **L4 — Tools**, nous reprenons les **outils** en profondeur : comment `@tool` construit le schéma que Mistral utilise pour décider *quand* et *comment* appeler une fonction.

## 📚 Documentation officielle

- [LangChain · Streaming](https://docs.langchain.com/oss/python/langchain/streaming)
- [LangChain · Agents (`create_agent`)](https://docs.langchain.com/oss/python/langchain/agents)
- [LangChain · Intégration ChatMistralAI](https://docs.langchain.com/oss/python/integrations/chat/mistralai)